# ***Assignment : Function***

In [1]:
import sqlite3
import pandas as pd
from IPython.display import display, HTML

# 1. DATABASE SETUP & DATA INSERTION [cite: 92-102]
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Creating the Student_Performance table [cite: 93-102]
cursor.execute("""
CREATE TABLE Student_Performance (
    student_id INT PRIMARY KEY,
    name VARCHAR(50),
    course VARCHAR(30),
    score INT,
    attendance INT,
    mentor VARCHAR(50),
    join_date DATE,
    city VARCHAR(50)
);
""")

# Inserting the Data (Correcting OCR errors from the PDF)
cursor.executescript("""
INSERT INTO Student_Performance VALUES
(101, 'Aarav Mehta', 'Data Science', 88, 92, 'Dr. Sharma', '2023-06-12', 'Mumbai'),
(102, 'Riya Singh', 'Data Science', 76, 85, 'Dr. Sharma', '2023-07-01', 'Delhi'),
(103, 'Kabir Khanna', 'Python', 91, 96, 'Ms. Nair', '2023-06-20', 'Mumbai'),
(104, 'Tanvi Patel', 'SQL', 84, 89, 'Mr. Iyer', '2023-05-30', 'Bengaluru'),
(105, 'Ayesha Khan', 'Python', 67, 81, 'Ms. Nair', '2023-07-18', 'Hyderabad'),
(106, 'Dev Sharma', 'SQL', 73, 78, 'Mr. Iyer', '2023-05-28', 'Pune'),
(107, 'Arjun Verma', 'Tableau', 95, 98, 'Ms. Kapoor', '2023-06-15', 'Delhi'),
(108, 'Meera Pillai', 'Tableau', 82, 87, 'Ms. Kapoor', '2023-06-18', 'Kochi'),
(109, 'Nikhil Rao', 'Data Science', 79, 82, 'Dr. Sharma', '2023-07-05', 'Chennai'),
(110, 'Priya Desai', 'SQL', 92, 94, 'Mr. Iyer', '2023-05-27', 'Bengaluru'),
(111, 'Siddharth Jain', 'Python', 85, 90, 'Ms. Nair', '2023-07-02', 'Mumbai'),
(112, 'Sneha Kulkarni', 'Tableau', 74, 83, 'Ms. Kapoor', '2023-06-10', 'Pune'),
(113, 'Rohan Gupta', 'SQL', 89, 91, 'Mr. Iyer', '2023-05-25', 'Delhi'),
(114, 'Ishita Joshi', 'Data Science', 93, 97, 'Dr. Sharma', '2023-06-25', 'Bengaluru'),
(115, 'Yuvraj Rao', 'Python', 71, 84, 'Ms. Nair', '2023-07-12', 'Hyderabad');
""")

def run_q(title, sql):
    print(f"\n{'='*15} {title} {'='*15}")
    display(pd.read_sql_query(sql, conn))

# ==========================================
# SQL FUNCTIONS ANSWERS [cite: 141-152]
# ==========================================

# Q1: Create a ranking of students based on score (highest first).
run_q("Q1: Student Ranking",
      "SELECT name, score, RANK() OVER(ORDER BY score DESC) as student_rank FROM Student_Performance")

# Q2: Show each student's score and the previous student's score (based on score order).
run_q("Q2: Score with Previous Score",
      "SELECT name, score, LAG(score) OVER(ORDER BY score DESC) as prev_score FROM Student_Performance")

# Q3: Convert names to uppercase and extract month name from join_date.
# (Using SQLite strftime for month extraction)
run_q("Q3: Uppercase and Month",
      """SELECT UPPER(name) as NAME_UPPER,
         CASE strftime('%m', join_date)
            WHEN '05' THEN 'May' WHEN '06' THEN 'June' WHEN '07' THEN 'July'
         END as join_month FROM Student_Performance""")

# Q4: Show each student's name and the next student's attendance (ordered by attendance).
run_q("Q4: Name and Next Attendance",
      "SELECT name, attendance, LEAD(attendance) OVER(ORDER BY attendance) as next_attendance FROM Student_Performance")

# Q5: Assign students into 4 performance groups using NTILE().
run_q("Q5: Performance Groups",
      "SELECT name, score, NTILE(4) OVER(ORDER BY score DESC) as performance_group FROM Student_Performance")

# Q6: For each course, assign a row number based on attendance (highest first).
run_q("Q6: Row Number by Course Attendance",
      "SELECT course, name, attendance, ROW_NUMBER() OVER(PARTITION BY course ORDER BY attendance DESC) as row_num FROM Student_Performance")

# Q7: Calculate enrollment days (from join_date to 2025-01-01).
run_q("Q7: Days Enrolled",
      "SELECT name, join_date, (julianday('2025-01-01') - julianday(join_date)) as days_enrolled FROM Student_Performance")

# Q8: Format join_date as "Month Year".
run_q("Q8: Format Date (Month Year)",
      """SELECT name,
         CASE strftime('%m', join_date)
            WHEN '05' THEN 'May' WHEN '06' THEN 'June' WHEN '07' THEN 'July'
         END || ' ' || strftime('%Y', join_date) as formatted_date FROM Student_Performance""")

# Q9: Replace 'Mumbai' with 'MUM' for display.
run_q("Q9: City Replace",
      "SELECT name, REPLACE(city, 'Mumbai', 'MUM') as city_short FROM Student_Performance")

# Q10: For each course, find the highest score using FIRST_VALUE().
run_q("Q10: Highest Score per Course",
      """SELECT DISTINCT course,
         FIRST_VALUE(score) OVER(PARTITION BY course ORDER BY score DESC) as top_course_score
         FROM Student_Performance""")


=============== Q1: Student Ranking ===============


,name,score,student_rank
0,Arjun Verma,95,1
1,Ishita Joshi,93,2
2,Priya Desai,92,3
3,Kabir Khanna,91,4
4,Rohan Gupta,89,5
5,Aarav Mehta,88,6
6,Siddharth Jain,85,7
7,Tanvi Patel,84,8
8,Meera Pillai,82,9
9,Nikhil Rao,79,10



=============== Q2: Score with Previous Score ===============


,name,score,prev_score
0,Arjun Verma,95,NaN
1,Ishita Joshi,93,95.0
2,Priya Desai,92,93.0
3,Kabir Khanna,91,92.0
4,Rohan Gupta,89,91.0
5,Aarav Mehta,88,89.0
6,Siddharth Jain,85,88.0
7,Tanvi Patel,84,85.0
8,Meera Pillai,82,84.0
9,Nikhil Rao,79,82.0



=============== Q3: Uppercase and Month ===============


...

,name,formatted_date
0,Aarav Mehta,June 2023
1,Riya Singh,July 2023
2,Kabir Khanna,June 2023
3,Tanvi Patel,May 2023
4,Ayesha Khan,July 2023
5,Dev Sharma,May 2023
6,Arjun Verma,June 2023
7,Meera Pillai,June 2023
8,Nikhil Rao,July 2023
9,Priya Desai,May 2023



=============== Q9: City Replace ===============


,name,city_short
0,Aarav Mehta,MUM
1,Riya Singh,Delhi
2,Kabir Khanna,MUM
3,Tanvi Patel,Bengaluru
4,Ayesha Khan,Hyderabad
5,Dev Sharma,Pune
6,Arjun Verma,Delhi
7,Meera Pillai,Kochi
8,Nikhil Rao,Chennai
9,Priya Desai,Bengaluru



=============== Q10: Highest Score per Course ===============


,course,top_course_score
0,Data Science,93
1,Python,91
2,SQL,92
3,Tableau,95
